In [32]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

In [33]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [34]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.6-flash")
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'}, google_api_key=SecretStr('**********'), model='gemini-3.6-flash', temperature=None, client=<google.genai.client.Client object at 0x0000029383AA36F0>, default_metadata=(), model_kwargs={})

In [35]:
#define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply 'a' and 'b'.
    
    Args:
        a (int): First int
        b (int): Second int
    """

    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


In [36]:
#augment the LLM with tools 
tools = [multiply, add, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)
model_with_tools

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'}, google_api_key=SecretStr('**********'), model='gemini-3.6-flash', temperature=None, client=<google.genai.client.Client object at 0x0000029383AA36F0>, default_metadata=

In [37]:
from langgraph.graph import add_messages
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    ToolCall,
)
from langchain_core.messages import BaseMessage
from langgraph.func import entrypoint, task


# 2. Define model node
* The model node is used to call the LLM and decide whether to call a tool or not.
* The @task decorator marks a function as a task that can be executed as part of the agent. Tasks can be called synchronously or asynchronously within your entrypoint function

In [38]:
@task
def call_llm(messages: list[BaseMessage]):
    """LLM decides whether to call a tool or not."""

    return model_with_tools.invoke(
        [
            SystemMessage(
                content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
            )
        ]
        + messages
    )

# 3. Define tool node
* The tool node is used to call the tools and return the results.

In [39]:
@task
def call_tools(tool_call: ToolCall):
    """Perfroms the tool call"""
    tool = tools_by_name[tool_call["name"]]
    return tool.invoke(tool_call)

# 4. Define agent
* The agent is built usinf the @entrypoint function


In [43]:
@entrypoint()
def agent(messages: list[BaseMessage]):
    model_response = call_llm(messages).result()
    # print(f"Model response: {model_response}")
    while True:
        if not model_response.tool_calls:
            break

        #Execute tool 
        tool_result_fututres = [
            call_tools(tool_call) for tool_call in model_response.tool_calls
        ]
        # print(f"Tool result futures: {tool_result_fututres}")
        tool_results = [fut.result() for fut in tool_result_fututres]
        # print(f"Tool results: {tool_results}")
        messages = add_messages(messages, [model_response, *tool_results])
        # print(f"Messages: {messages}")
        model_response = call_llm(messages).result()
        # print(f"Model response: {model_response}")

    messages = add_messages(messages, model_response)
    return messages

#invoke
messages = [HumanMessage(content="Add 3 and 4.")]
stream = agent.invoke(messages)
for snapshot in stream:
    # print(f"Snapshot: {snapshot}")
    # print("\n\n")
    snapshot.pretty_print()

================================ Human Message =================================

Add 3 and 4.
================================== Ai Message ==================================

[]
Tool Calls:
  add (call_800964)
 Call ID: call_800964
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: add

7
================================== Ai Message ==================================

[{'type': 'text', 'text': 'The sum of 3 and 4 is 7.', 'extras': {'signature': 'EqkBCqYBARFNMg+x/BkYapr1cxJYGzxdrB8z7OdwqyKfGhngZYpYLpIsKbswaLQt6RdsHSV0DXrdWQmpJBtKDhJ9cp7jx9gQaYqRhGDr6vauQEgDgtxCf+ss9BI0rP4Aftji24CVPvLStBdxbbjQoWMJV1/CLLlGOwb6lU/fj8UJaIPxtUKJSyVDr4E4Zj3Dgm3lbjKFB7NA+o6I8xOQsFMFdPWWEwGz/OYbTA=='}}]
